# Question 1

In [7]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

In [19]:
chars = ['g', 'o', 'd']
V= len(chars)  
H = 2                   
char2idx ={c: i for i, c in enumerate(chars)}
idx2char= {i: c for i, c in enumerate(chars)}
print(f"Vocabulary : {chars}  (V={V})")
print(f"Hidden size: H = {H}")
print(char2idx)
print(idx2char)

Vocabulary : ['g', 'o', 'd']  (V=3)
Hidden size: H = 2
{'g': 0, 'o': 1, 'd': 2}
{0: 'g', 1: 'o', 2: 'd'}


In [20]:
u = np.array([[0.1, 0.2, 0.3],   # shape (H, V)
                 [0.4, 0.5, 0.6]])

w = np.array([[0.1, 0.2],     # shape (H, H)
                 [0.3, 0.4]])

v = np.array([[0.1, 0.2],       # shape (V, H)
                 [0.3, 0.4],
                 [0.5, 0.6]])

b  = np.zeros((H, 1))       # hidden bias
c  = np.zeros((V, 1))       # output bias
h0  = np.zeros((H, 1))      # initial hidden state

#simple manual enccoding
word    = "good"
inputs  = list(word[:-1])   # g, o, o  — characters fed in
targets = list(word[1:])    # o, o, d  — characters to predict
T       = len(inputs)        # time steps = 3
def one_hot(char):
    vec = np.zeros((V, 1))
    vec[char2idx[char]] = 1.0
    return vec
X = [one_hot(c) for c in inputs]
Y = [one_hot(c) for c in targets]
print(inputs)
print(targets)
print(X)
print(Y)

['g', 'o', 'o']
['o', 'o', 'd']
[array([[1.],
       [0.],
       [0.]]), array([[0.],
       [1.],
       [0.]]), array([[0.],
       [1.],
       [0.]])]
[array([[0.],
       [1.],
       [0.]]), array([[0.],
       [1.],
       [0.]]), array([[0.],
       [0.],
       [1.]])]


In [10]:
def tanh(a):
    return np.tanh(a)

def tanh_deriv(h):
    return 1.0 - h ** 2

def softmax(z):
    e = np.exp(z - np.max(z)) #for stability max is subtracted
    return e / e.sum()

H_states = [h0]   
Y_hats   = []     
Z_logits = []    
for t in range(T):
    h_prev = H_states[t]
    x_t    = X[t]
    z_t = w @ h_prev + u @ x_t + b   # shape (H,1)
    a_t = tanh(z_t)                  # shape (H,1)
    o_t = v @ a_t + c                # shape (V,1)
    y_hat_t = softmax(o_t)           # shape (V,1)
    H_states.append(a_t)
    Z_logits.append(o_t)
    Y_hats.append(y_hat_t)

print(H_states)
print(Z_logits)
print(Y_hats)

[array([[0.],
       [0.]]), array([[0.0997],
       [0.3799]]), array([[0.2784],
       [0.5927]]), array([[0.3332],
       [0.6754]])]
[array([[0.086 ],
       [0.1819],
       [0.2778]]), array([[0.1464],
       [0.3206],
       [0.4948]]), array([[0.1684],
       [0.3701],
       [0.5718]])]
[array([[0.3019],
       [0.3323],
       [0.3658]]), array([[0.2772],
       [0.33  ],
       [0.3928]]), array([[0.2688],
       [0.3289],
       [0.4024]])]


In [11]:
def cross_entropy(y_hat, y):
    return -np.sum(y * np.log(y_hat + 1e-12))
total_loss = 0.0
for t in range(T):
    L_t = cross_entropy(Y_hats[t], Y[t])
    total_loss += L_t
    correct_prob = Y_hats[t][np.argmax(Y[t]), 0]
    print(f"  L^({t+1}) = −log(ŷ_{targets[t]}^({t+1})) = −log({correct_prob:.4f}) = {L_t:.4f}")
print(f"  Total Loss L = {' + '.join([f'L^({t+1})' for t in range(T)])} = {total_loss:.4f}")

  L^(1) = −log(ŷ_o^(1)) = −log(0.3323) = 1.1017
  L^(2) = −log(ŷ_o^(2)) = −log(0.3300) = 1.1087
  L^(3) = −log(ŷ_d^(3)) = −log(0.4024) = 0.9104
  Total Loss L = L^(1) + L^(2) + L^(3) = 3.1208


In [12]:
du = np.zeros_like(u)
dw = np.zeros_like(w)
dv = np.zeros_like(v)
db  = np.zeros_like(b)
dc  = np.zeros_like(c)

delta_a_next = np

np.zeros((H, 1))   # delta_a^(t+1), zero for t = T
for t in reversed(range(T)):
    delta_y = Y_hats[t] - Y[t]                      
    dv += delta_y @ H_states[t+1].T                 # (V,1)·(1,H) = (V,H)
    dc  += delta_y
    delta_h = v.T @ delta_y + w.T @ delta_a_next  # (H,1)
    delta_a = delta_h * tanh_deriv(H_states[t+1])     # (H,1)
    du += delta_a @ X[t].T                           # (H,1)·(1,V) = (H,V)
    dw += delta_a @ H_states[t].T                    # (H,1)·(1,H) = (H,H)
    db  += delta_a
    delta_a_next = delta_a   #Backprop

In [13]:
delta_a_next

array([[ 0.002 ],
       [-0.0024]])

In [18]:
#simple SGD updates
lr = 0.1
u_new = u - lr * du
w_new = w - lr * dw
v_new = v - lr * dv
b_new  = b  - lr * db
c_new  = c  - lr * dc
for t in range(1, T+1):
    print(f"h^({t}) = {H_states[t].ravel()}")
print(f"Predictions:")
for t in range(T):
    pred = idx2char[np.argmax(Y_hats[t])]
    corr = 'Correct' if pred == targets[t] else 'wrong'
    print(f"t={t+1}: '{inputs[t]}' → predicted='{pred}'  target='{targets[t]}'  {corr}")
print()

h^(1) = [0.0997 0.3799]
h^(2) = [0.2784 0.5927]
h^(3) = [0.3332 0.6754]
Predictions:
t=1: 'g' → predicted='d'  target='o'  wrong
t=2: 'o' → predicted='d'  target='o'  wrong
t=3: 'o' → predicted='d'  target='d'  Correct



# Question 2

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [23]:
WORDS = [
    "apple", "bat", "cat", "dog", "elephant", "girl", "hen",
    "ice", "joy", "king", "lemon", "nest", "orange", "parrot",
    "queen", "rat", "sun", "tiger", "umbrella", "van", "watch",
    "xylophone", "yard", "zebra"
]

SOS, EOS = '<', '>'
all_chars  = sorted(set("".join(WORDS) + SOS + EOS))
V          = len(all_chars)                            # vocabulary size = 27
char2idx   = {c: i for i, c in enumerate(all_chars)}
idx2char   = {i: c for i, c in enumerate(all_chars)}

print(f"  Size           : V = {V}")
print(f"  Characters     : {all_chars}")
print(f"  char→index     : e.g.  'a'→{char2idx['a']},  'z'→{char2idx['z']}")



  Size           : V = 27
  Characters     : ['<', '>', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
  char→index     : e.g.  'a'→2,  'z'→26


In [25]:
def one_hot(idx, vocab_size=V):
    v = torch.zeros(vocab_size)
    v[idx] = 1.0
    return v

def encode_word(word):
    seq     = SOS + word + EOS
    inputs  = torch.stack([one_hot(char2idx[c]) for c in seq[:-1]])  # [T, V]
    targets = torch.tensor([char2idx[c] for c in seq[1:]], dtype=torch.long)  # [T]
    return inputs, targets

In [27]:
class WordDataset(Dataset):
    def __init__(self, words):
        self.samples = [encode_word(w) for w in words]
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]
def collate_fn(batch):
    inputs, targets = zip(*batch)
    lengths = torch.tensor([x.shape[0] for x in inputs])
    T_max   = lengths.max().item()
    pad_inputs  = torch.zeros(len(inputs), T_max, V)
    pad_targets = torch.zeros(len(inputs), T_max, dtype=torch.long)
    for i, (x, y) in enumerate(zip(inputs, targets)):
        t = x.shape[0]
        pad_inputs[i, :t, :]  = x
        pad_targets[i, :t]    = y
    return pad_inputs, pad_targets, lengths


In [28]:
dataset    = WordDataset(WORDS)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [29]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size=V, hidden_size=64):
        super().__init__()
        self.hidden_size = hidden_size
        # Core RNN: input_size=V, hidden_size=H
        # batch_first=True->input shape [batch, seq_len, input_size]
        self.rnn = nn.RNN(
            input_size  = vocab_size,
            hidden_size = hidden_size,
            num_layers  = 1,
            batch_first = True,
            nonlinearity= 'tanh'       # tanh activation (default)
        )

        # Fully-connected output layer: H → V (logits over vocabulary)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h0=None):
        # out : [batch, seq_len, H]
        # h_n : [1,     batch,   H]
        out, h_n = self.rnn(x, h0)
        # Project every timestep to vocv size
        # [batch, seq_len, H] → [batch, seq_len, V]
        logits = self.fc(out)
        return logits, h_n

    def predict_next(self, char, h=None):
        self.eval()
        with torch.no_grad():
            x      = one_hot(char2idx[char]).unsqueeze(0).unsqueeze(0)  # [1,1,V]
            logits, h = self.forward(x, h)
            probs  = torch.softmax(logits[0, 0], dim=0)
            pred   = idx2char[probs.argmax().item()]
        return pred, h

In [30]:
model = SimpleRNN(vocab_size=V, hidden_size=64)

In [31]:
print(model)

SimpleRNN(
  (rnn): RNN(27, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=27, bias=True)
)


In [ ]:
def masked_cross_entropy(logits, targets, lengths):
    B, T, V_out = logits.shape
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    # Flatten: [B*T, V] and [B*T]
    loss_all = loss_fn(logits.reshape(B * T, V_out),
                       targets.reshape(B * T))          # [B*T]
    loss_all = loss_all.reshape(B, T)                   # [B, T]
    # Build mask: 1 for real positions, 0 for padding
    mask = torch.zeros(B, T)
    for i, l in enumerate(lengths):
        mask[i, :l] = 1.0
    # Average over real positions only
    return (loss_all * mask).sum() / mask.sum()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-2)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

EPOCHS = 150
history = []

In [ ]:
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0

    for batch_x, batch_y, lengths in dataloader:
        optimizer.zero_grad()
        logits, _ = model(batch_x)                           # [B, T, V]
        loss      = masked_cross_entropy(logits, batch_y, lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)    # gradient clipping
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg = epoch_loss / len(dataloader)
    history.append(avg)

    if epoch % 25 == 0 or epoch == 1:
        # Quick prediction demo
        h = None
        pred_word = ""
        ch = SOS
        for _ in range(8):
            ch, h = model.predict_next(ch, h)
            if ch == EOS:
                break
            pred_word += ch
        print(f"{epoch:>6}   {avg:>9.4f}  generated from '<': '{pred_word}'")

print(f"\nFinal loss: {history[-1]:.4f}")

In [ ]:
model.eval()
total_correct = 0
total_chars   = 0

with torch.no_grad():
    for word in WORDS:
        inputs, targets = encode_word(word)        # [T,V], [T]
        x      = inputs.unsqueeze(0)               # [1, T, V]
        logits, _ = model(x)                       # [1, T, V]
        preds  = logits[0].argmax(dim=-1)          # [T]

        pred_str   = "".join(idx2char[i.item()] for i in preds)
        target_str = "".join(idx2char[i.item()] for i in targets)
        correct    = (preds == targets).sum().item()
        total_correct += correct
        total_chars   += len(targets)
        acc = correct / len(targets) * 100

        seq_in = SOS + word
        print(f"{word:<12} {seq_in:<12} {pred_str:<12} {target_str:<12} {acc:5.1f}%")

print(f"{'OVERALL':<36} {total_correct}/{total_chars}  {total_correct/total_chars*100:.1f}%")

In [ ]:
model.eval()
with torch.no_grad():
    for word in WORDS[:10]:   # show first 10 words
        h  = None
        ch = SOS
        generated = ""
        for _ in range(12):
            ch, h = model.predict_next(ch, h)
            if ch == EOS:
                break
            generated += ch
        status = "✓" if generated == word else "✗"
        print(f"  {word:<12} '<'      → '{generated}'  {status}")

# Question 3

In [2]:
import numpy as np

A = np.array([[1, 1, 0, 0],
              [1, 1, 1, 0],
              [0, 1, 1, 1],
              [0, 0, 1, 1]], dtype=float)
X = np.array([[1, 0],
              [0, 1],
              [1, 1],
              [0, 0]], dtype=float)
W = np.array([[1, -1],
              [0,  1]], dtype=float)
AX  = A @ X
AXW = AX @ W
H1  = np.maximum(0, AXW)   # ReLU 

print("AX  =\n", AX)
print("AXW =\n", AXW)
print("H^(1) = ReLU(AXW) =\n", H1)

AX  =
 [[1. 1.]
 [2. 2.]
 [1. 2.]
 [1. 1.]]
AXW =
 [[1. 0.]
 [2. 0.]
 [1. 1.]
 [1. 0.]]
H^(1) = ReLU(AXW) =
 [[1. 0.]
 [2. 0.]
 [1. 1.]
 [1. 0.]]
